# Notebook 09r — cứu dự đoán L2 từ checkpoint tốt nhất

Chạy trong **cùng kernel** ngay sau notebook 09 (lượt 2) khi ô train dừng ở assert quá chặt; không train lại.

In [ ]:
TEXT_COL = "text_L2"

In [ ]:
# ===== CỨU DỰ ĐOÁN TỪ CHECKPOINT TỐT NHẤT (chạy trong cùng kernel sau khi ô train dừng ở assert quá chặt) =====
# Ô train trước đã train xong 3 epoch và chọn epoch tốt nhất, nhưng dừng ở assert "Macro-F1 cuối == best_metric" (dung sai 1e-6
# quá chặt vì padding khác nhau giữa lúc train và lúc dự đoán làm bf16 lệch nhẹ). Checkpoint tốt nhất vẫn còn trên đĩa.
import glob
from transformers import AutoModelForSequenceClassification, AutoTokenizer
ckpts = sorted(glob.glob(f"{WORK}/ckpt_{KEY}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
state = json.load(open(f"{ckpts[-1]}/trainer_state.json", encoding="utf-8"))
best_dir, best_metric = state["best_model_checkpoint"], state["best_metric"]
hist = [h for h in state["log_history"] if "eval_macro_f1" in h]
log(f"Cứu từ {best_dir} | best_metric lúc train {best_metric:.4f} | các epoch: "
    + ", ".join(f"{h['eval_macro_f1']:.4f}" for h in hist))
tok = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(best_dir).cuda().eval()
texts = list(val_df[TEXT_COL])
out = []
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16):
    for i in range(0, len(texts), 128):                      # đúng thứ tự val_df
        enc = tok(texts[i:i + 128], truncation=True, max_length=CFG["max_length"], padding=True, return_tensors="pt").to("cuda")
        out.append(torch.softmax(model(**enc).logits.float(), -1).cpu().numpy())
PROBS = np.concatenate(out)
gap = metrics_dict(val_df["label"].values, PROBS.argmax(1))["macro_f1"] - best_metric
log(f"Macro-F1 dự đoán tuần tự − best_metric lúc train = {gap:+.5f}")
assert abs(gap) < 0.005
json.dump({"name": KEY, "cfg": CFG, "epoch_evals": hist, "best_metric": best_metric, "best_checkpoint": best_dir,
           "gpu": GPU, "transformers": TV, "recovered_from_checkpoint": True, "quick": QUICK},
          open(f"{OUT_DIR}/train_history_{KEY}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

[2026-10-31 06:37:24 UTC] [L2] Cứu từ /content/ckpt_L2/checkpoint-7200 | best_metric lúc train 0.5784 | các epoch: 0.5681, 0.5784


/usr/local/lib/python3.13/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[2026-10-31 06:43:26 UTC] [L2] Macro-F1 dự đoán tuần tự − best_metric lúc train = +0.00007


In [ ]:
save_predictions(PROBS, KEY)
k = val_df["source_kind"].values
CMP = compare_with_reference(PROBS.argmax(1), KEY, subsets={
    "all": np.ones(len(k), bool),
    "source: full text (2 đề)": k == "full_text",
    "source: GPT-4 summary (2 đề)": k == "gpt4_summary",
    "source: any (4 đề)": k != "none",
    "no source (11 đề, đầu vào = L1)": k == "none",
})
# Hiệu ứng của nguồn = chênh (L2 − L1) ở đề có nguồn − chênh ở đề không có nguồn, bootstrap theo bài
ref1 = REF.set_index("row_idx").loc[val_df["row_idx"], "pred_label"].values
y, ess, pred = val_df["label"].values, val_df["essay_id_comp"].values, PROBS.argmax(1)
src, nos = k != "none", k == "none"
codes = pd.Series(ess).astype("category").cat.codes.values
members = [np.flatnonzero(codes == c) for c in range(codes.max() + 1)]
rng = np.random.default_rng(7); dd = []
for _ in range(200 if QUICK else 2000):
    idx = np.concatenate([members[c] for c in rng.integers(0, len(members), len(members))])
    a, b = idx[src[idx]], idx[nos[idx]]
    dd.append((macro(y[a], pred[a]) - macro(y[a], ref1[a])) - (macro(y[b], pred[b]) - macro(y[b], ref1[b])))
did = {"did_source_effect": round((macro(y[src], pred[src]) - macro(y[src], ref1[src]))
                                  - (macro(y[nos], pred[nos]) - macro(y[nos], ref1[nos])), 4),
       "ci_low": round(np.percentile(dd, 2.5), 4), "ci_high": round(np.percentile(dd, 97.5), 4)}
json.dump(did, open(f"{OUT_DIR}/source_effect_diff_in_diff.json", "w"), indent=2)
log(f"Hiệu ứng nguồn (diff-in-diff vs L1): {did}")
# Theo loại đơn vị trên đề có nguồn (Evidence là nơi giả thuyết dự đoán có tác dụng)
rows = []
for t in sorted(val_df.discourse_type.unique()):
    m = src & (val_df.discourse_type.values == t)
    rows.append({"discourse_type": t, "n": int(m.sum()), "L2": round(macro(y[m], pred[m]), 4),
                 "L1": round(macro(y[m], ref1[m]), 4), "delta": round(macro(y[m], pred[m]) - macro(y[m], ref1[m]), 4)})
by_type = pd.DataFrame(rows); by_type.to_csv(f"{OUT_DIR}/L2_vs_L1_by_type_source_prompts.csv", index=False)
print(by_type.to_string(index=False))

[2026-10-31 06:43:26 UTC] [L2] Lưu /content/out_L2/val_predictions_L2.csv


model                          subset     baseline  macro_f1_model  macro_f1_baseline   delta  ci_low  ci_high  p_one_sided_le0  n_units  n_essays  n_boot
   L2                             all L1 (tuần 11)          0.5785             0.5809 -0.0024 -0.0096   0.0047           0.7495    29102      3119    2000
   L2                             all L0 (tuần 11)          0.5785             0.5723  0.0062 -0.0015   0.0143           0.0580    29102      3119    2000
   L2        source: full text (2 đề) L1 (tuần 11)          0.5492             0.5561 -0.0070 -0.0207   0.0060           0.8410     5084       561    2000
   L2        source: full text (2 đề) L0 (tuần 11)          0.5492             0.5543 -0.0051 -0.0209   0.0109           0.7365     5084       561    2000
   L2    source: GPT-4 summary (2 đề) L1 (tuần 11)          0.4985             0.5082 -0.0097 -0.0391   0.0200           0.7195     2561       301    2000
   L2    source: GPT-4 summary (2 đề) L0 (tuần 11)          0.4985    

[2026-10-31 06:45:34 UTC] [L2] Hiệu ứng nguồn (diff-in-diff vs L1): {'did_source_effect': -0.0139, 'ci_low': np.float64(-0.0294), 'ci_high': np.float64(0.002)}


      discourse_type    n     L2     L1   delta
               Claim 2477 0.5014 0.5016 -0.0002
Concluding Statement  721 0.5912 0.5741  0.0170
        Counterclaim  336 0.5109 0.4816  0.0294
            Evidence 2513 0.5538 0.5934 -0.0396
                Lead  498 0.5027 0.5598 -0.0571
            Position  840 0.5194 0.4597  0.0596
            Rebuttal  260 0.5300 0.4995  0.0305


In [ ]:
# ===== XUẤT KẾT QUẢ =====
m = metrics_dict(val_df["label"].values, PROBS.argmax(1))
summary = {"model": KEY, **{k: round(float(v), 4) for k, v in m.items()}, "n_val_units": len(val_df),
           "quick": QUICK, "gpu": GPU, "transformers": TV, "finished_at": datetime.now(timezone.utc).isoformat()}
json.dump(summary, open(f"{OUT_DIR}/summary_{KEY}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
log(f"KẾT QUẢ {KEY} (validation): " + " | ".join(f"{k} {v}" for k, v in summary.items() if isinstance(v, float)))
bundle = f"{WORK}/results_{KEY}" + ("_QUICK" if QUICK else "") + ".zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        zf.write(f"{OUT_DIR}/{fn}", f"{os.path.basename(OUT_DIR)}/{fn}")
print("Đã đóng gói:", bundle, os.path.getsize(bundle), "bytes")

[2026-10-31 06:45:34 UTC] [L2] KẾT QUẢ L2 (validation): macro_f1 0.5785 | accuracy 0.7308 | balanced_accuracy 0.6557 | qwk 0.4538 | f1_ineffective 0.3776 | f1_adequate 0.8167 | f1_effective 0.5412 | recall_ineffective 0.5444 | precision_ineffective 0.2891


Đã đóng gói: /content/results_L2.zip 529242 bytes
